# Montar o app ExB "Mobilidade Urbana" (app 3) — v4
**Novidades da v4**
- Cria o widget próprio **Horários da Linha** em `client\your-extensions\widgets\horarios-linha` (mesmo padrão do Placar das câmeras):
  cartão escuro como no protótipo — nº da linha, nome, chips *Dias úteis* / sentido, **próxima saída em destaque com relógio**,
  todos os horários por ponto (os que já passaram ficam apagados) e plano B quando o sentido não tem horários próprios.
- Coluna direita: Horários da Linha (grande) + mapa de concentração pequeno embaixo (com tela cheia). Botão **Leia-me** no topo do mapa.
- Casca do CCTV mantida.

⚠ **Depois de rodar:** reinicie o *client* do ExB (widget novo) → no construtor **Ctrl+F5 → mudança pequena → Salvar → Publicar**.
Saída: `Documents\Saidas_Notebooks\Montar_ExB_Mobilidade_v4.txt`

In [ ]:
# ===== CÉLULA 1 — Configuração =====
import os, json, datetime, shutil, copy
NB_NOME = "Montar_ExB_Mobilidade_v4"
EXB = r"C:\ExB_1.19"
EXB_APPS = os.path.join(EXB, "server", "public", "apps")
WIDGETS_DIR = os.path.join(EXB, "client", "your-extensions", "widgets")
APP_MODELO, APP_NOVO = "2", "3"
CFG_MODELO = os.path.join(EXB_APPS, APP_MODELO, "resources", "config", "config.json")
CFG_NOVO = os.path.join(EXB_APPS, APP_NOVO, "resources", "config", "config.json")
home = os.path.expanduser("~")
_c = [os.path.join(home, "Documents", "Saidas_Notebooks"), os.path.join(home, "OneDrive", "Documents", "Saidas_Notebooks")]
PASTA_SAIDA = next((c for c in _c if os.path.isdir(c)), _c[0]); os.makedirs(PASTA_SAIDA, exist_ok=True)
PASTA_BKP = os.path.join(PASTA_SAIDA, f"backup_exb_mobilidade_{datetime.datetime.now():%Y%m%d_%H%M%S}")
os.makedirs(PASTA_BKP, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")
def log(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f: f.write(s + "\n")
log("▶ CÉLULA 1 — Configuração: iniciando...")
for p in (CFG_MODELO, CFG_NOVO, WIDGETS_DIR):
    if not os.path.exists(p): raise RuntimeError("Não encontrado: " + p)
log("  Backups:", PASTA_BKP)
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — Widget próprio 'Horários da Linha' =====
log("\n▶ CÉLULA 2 — Criação do widget Horários da Linha: iniciando...")
ARQUIVOS_WIDGET = {
    'manifest.json': r'''{
  "name": "horarios-linha",
  "label": "Horários da Linha",
  "type": "widget",
  "version": "1.19.0",
  "exbVersion": "1.19.0",
  "author": "GeoAPUC",
  "description": "Mostra a tabela de horários (dias úteis) da linha de ônibus selecionada na lista, com a próxima saída em destaque.",
  "copyright": "",
  "license": "",
  "properties": {},
  "translatedLocales": ["en"],
  "defaultSize": { "width": 380, "height": 560 }
}
''',
    'config.json': r'''{
  "dataSourceId": "dataSource_1-linhas_onibus",
  "linhasUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Linhas_Onibus/FeatureServer/0",
  "horariosUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Horarios_Onibus/FeatureServer/0",
  "tema": "escuro"
}
''',
    'icon.svg': r'''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 16 16" width="16" height="16"><rect x="3" y="1.5" width="10" height="11" rx="2" fill="none" stroke="#000" stroke-width="1.2"/><rect x="4.6" y="3.2" width="6.8" height="3.6" rx=".6" fill="#000"/><circle cx="5.5" cy="9.6" r=".9" fill="#000"/><circle cx="10.5" cy="9.6" r=".9" fill="#000"/><rect x="4" y="12.5" width="2" height="2" rx=".5" fill="#000"/><rect x="10" y="12.5" width="2" height="2" rx=".5" fill="#000"/></svg>
''',
    'src/runtime/widget.tsx': r'''// Horários da Linha – GeoAPUC (v1)
// Mostra os horários (dias úteis) da linha selecionada na lista de linhas.
// Escrito sem JSX (React.createElement) para não depender da configuração de compilação.
import { React, ReactRedux, type AllWidgetProps } from 'jimu-core'

const h = React.createElement
const { useState, useEffect } = React

const TEMAS: any = {
  escuro: { fundo: '#0f1e33', card: '#16294a', borda: 'rgba(255,255,255,0.10)', texto: '#ffffff', suave: '#9fb0cc',
            destaque: '#F5A623', azul: '#4285F4', verde: '#1D9E75', chip: '#1f3b66', aviso: 'rgba(245,166,35,0.14)' },
  claro: { fundo: '#ffffff', card: '#f4f6fb', borda: '#dfe5ef', texto: '#1b2a4a', suave: '#6b7690',
           destaque: '#4463a9', azul: '#4463a9', verde: '#1f9d55', chip: '#e6ecf8', aviso: '#FFF4E0' }
}

function hhmm (v: any): string {
  const s = String(v ?? '').trim()
  const m = s.match(/^(\d{1,2}):(\d{2})/)
  return m ? `${m[1].padStart(2, '0')}:${m[2]}` : s
}
function minutos (v: string): number {
  const m = v.match(/^(\d{2}):(\d{2})/)
  return m ? Number(m[1]) * 60 + Number(m[2]) : -1
}
function aspas (v: any): string { return String(v ?? '').replace(/'/g, "''") }

async function consultar (url: string, where: string, campos: string): Promise<any[]> {
  const q = `${url}/query?where=${encodeURIComponent(where)}&outFields=${campos}&returnGeometry=false&f=json`
  const j: any = await fetch(q).then(r => r.json())
  if (j.error) throw new Error(j.error.message || 'erro na consulta')
  return (j.features || []).map((f: any) => f.attributes || {})
}

function relogio (cor: string, tam: number) {
  return h('svg', { width: tam, height: tam, viewBox: '0 0 24 24', style: { flex: 'none' } },
    h('circle', { cx: 12, cy: 12, r: 10, fill: '#ffffff', stroke: cor, strokeWidth: 2 }),
    h('path', { d: 'M12 6v6l4 2', fill: 'none', stroke: '#c0392b', strokeWidth: 2, strokeLinecap: 'round' }))
}

export default function Widget (props: AllWidgetProps<any>) {
  const cfg: any = props.config || {}
  const dsId: string = cfg.dataSourceId || 'dataSource_1-linhas_onibus'
  const urlLin: string = cfg.linhasUrl
  const urlHor: string = cfg.horariosUrl
  const T: any = TEMAS[cfg.tema] || TEMAS.escuro

  // id do registro selecionado na lista (fica no estado do app)
  const oid: string | null = ReactRedux.useSelector((s: any) => {
    const inf = s?.dataSourcesInfo || {}
    for (const k of Object.keys(inf)) {
      if (k === dsId || k.startsWith(dsId + '-')) {
        const ids = inf[k]?.selectedIds
        if (ids && ids.length) return String(ids[0])
      }
    }
    return null
  })

  const [linha, setLinha] = useState<any>(null)
  const [hor, setHor] = useState<any[]>([])
  const [planoB, setPlanoB] = useState<boolean>(false)
  const [carregando, setCarregando] = useState<boolean>(false)
  const [erro, setErro] = useState<string | null>(null)
  const [agora, setAgora] = useState<Date>(new Date())

  useEffect(() => { const t = setInterval(() => setAgora(new Date()), 60000); return () => clearInterval(t) }, [])

  useEffect(() => {
    let vivo = true
    if (!oid) { setLinha(null); setHor([]); return }
    setCarregando(true); setErro(null)
    ;(async () => {
      try {
        const l = (await consultar(urlLin, `objectid=${Number(oid)}`, 'objectid,id_rota,cod_linha,nome,sentido'))[0]
        if (!l) throw new Error('linha não encontrada')
        let rows = await consultar(urlHor, `id_rota='${aspas(l.id_rota)}'`, 'id_rota,ponto,horario,sentido,observacao')
        let b = false
        if (!rows.length) {
          const cod = String(l.id_rota || '').split('_')[0]
          rows = (await consultar(urlHor, `id_rota LIKE '${aspas(cod)}%'`, 'id_rota,ponto,horario,sentido,observacao'))
            .filter((r: any) => String(r.id_rota || '').startsWith(cod + '_'))
          b = rows.length > 0
        }
        rows.forEach((r: any) => { r._h = hhmm(r.horario); r._m = minutos(r._h) })
        rows.sort((a: any, z: any) => a._m - z._m)
        if (vivo) { setLinha(l); setHor(rows); setPlanoB(b); setCarregando(false) }
      } catch (e: any) {
        console.warn('[Horários da Linha]', e)
        if (vivo) { setErro('Não foi possível carregar os horários.'); setCarregando(false) }
      }
    })()
    return () => { vivo = false }
  }, [oid, urlLin, urlHor])

  const chip = (txt: string, bg: string, cor = '#ffffff') =>
    h('span', { style: { display: 'inline-block', padding: '2px 10px', borderRadius: 999, background: bg, color: cor, fontSize: 11, fontWeight: 700 } }, txt)
  const rotulo = (txt: string) => h('div', { style: { fontSize: 10, letterSpacing: 0.6, color: T.suave, textTransform: 'uppercase' } }, txt)

  const caixa = (filhos: any[]) => h('div', {
    className: 'jimu-widget horarios-linha',
    style: { width: '100%', height: '100%', boxSizing: 'border-box', background: T.fundo, color: T.texto, borderRadius: 12,
             padding: 14, overflowY: 'auto', fontFamily: 'inherit', display: 'flex', flexDirection: 'column', gap: 12,
             border: `1px solid ${T.borda}` }
  }, h('div', { style: { fontSize: 15, fontWeight: 600, color: T.texto } }, 'Tabela de Horários'), ...filhos)

  if (!oid) {
    return caixa([h('div', { style: { margin: 'auto', textAlign: 'center', color: T.suave, fontSize: 13, lineHeight: 1.5 } },
      h('div', { style: { fontSize: 34, marginBottom: 6 } }, '🚌'), 'Selecione uma linha na lista', h('br'), 'para ver os horários.')])
  }
  if (carregando) return caixa([h('div', { style: { margin: 'auto', color: T.suave } }, 'Carregando horários…')])
  if (erro || !linha) return caixa([h('div', { style: { margin: 'auto', color: '#e57373' } }, erro || '—')])

  // próxima saída (só em dias úteis)
  const dow = agora.getDay()
  const diaUtil = dow >= 1 && dow <= 5
  const minAgora = agora.getHours() * 60 + agora.getMinutes()
  const proxima = diaUtil ? hor.find((r: any) => r._m >= minAgora) : null
  let destaque: string = proxima ? proxima._h : (hor[0]?._h || '—')
  let sub = ''
  if (!hor.length) sub = 'Sem horários cadastrados para esta linha.'
  else if (!diaUtil) sub = 'Fim de semana — horários válidos para dias úteis. Primeira saída:'
  else if (proxima) { const d = proxima._m - minAgora; sub = d === 0 ? 'Saindo agora' : `Próxima saída · em ${d >= 60 ? Math.floor(d / 60) + ' h ' : ''}${d % 60} min` }
  else sub = 'Não há mais saídas hoje. Primeira saída amanhã:'

  // agrupa por sentido (plano B) e por ponto
  const grupos: any = {}
  hor.forEach((r: any) => {
    const k = (planoB ? (r.sentido || 'Outro sentido') + ' · ' : '') + (r.ponto || '—')
    ;(grupos[k] = grupos[k] || []).push(r)
  })
  const obs = Array.from(new Set(hor.map((r: any) => String(r.observacao || '').trim()).filter((x: string) => x && x !== '-')))

  return caixa([
    // cabeçalho da linha
    h('div', { style: { display: 'flex', alignItems: 'center', gap: 12, background: T.card, borderRadius: 10, padding: 10 } },
      h('div', { style: { minWidth: 46, textAlign: 'center', padding: '8px 6px', borderRadius: 8, border: `1px solid ${T.destaque}`,
                          color: T.destaque, fontWeight: 800, fontSize: 17, background: 'rgba(245,166,35,0.10)' } }, String(linha.cod_linha ?? '')),
      h('div', { style: { display: 'flex', flexDirection: 'column', gap: 4, minWidth: 0 } },
        h('div', { style: { fontSize: 16, fontWeight: 700, lineHeight: 1.2 } }, linha.nome || ''),
        h('div', { style: { display: 'flex', gap: 6, flexWrap: 'wrap' } }, chip('Dias úteis', T.azul), chip(linha.sentido || '', T.verde)))),
    // próxima saída
    h('div', { style: { display: 'flex', alignItems: 'center', gap: 12, background: T.card, borderRadius: 10, padding: '10px 12px' } },
      relogio(T.suave, 34),
      h('div', null, rotulo(sub || 'Horário'), h('div', { style: { fontSize: 32, fontWeight: 800, lineHeight: 1.1 } }, destaque))),
    planoB
      ? h('div', { style: { background: T.aviso, borderLeft: `4px solid ${T.destaque}`, borderRadius: 6, padding: '6px 10px', fontSize: 12, color: T.texto } },
          `Este sentido não tem horários próprios. Abaixo, os horários de saída da linha ${linha.cod_linha} nos outros sentidos.`)
      : null,
    // todos os horários
    ...Object.keys(grupos).map((k: string) => h('div', { key: k, style: { display: 'flex', flexDirection: 'column', gap: 6 } },
      rotulo('Ponto · ' + k),
      h('div', { style: { display: 'flex', flexWrap: 'wrap', gap: 6 } },
        ...grupos[k].map((r: any, i: number) => {
          const passou = diaUtil && r._m < minAgora
          const ehProx = proxima && r === proxima
          return h('span', { key: i, style: { padding: '4px 9px', borderRadius: 6, fontWeight: 700, fontSize: 13,
            background: ehProx ? T.destaque : T.chip, color: ehProx ? '#1b2a4a' : T.texto, opacity: passou ? 0.4 : 1 } }, r._h)
        })))),
    obs.length ? h('div', null, rotulo('Observação'), h('div', { style: { fontSize: 12, color: T.suave } }, obs.join(' · '))) : null,
    h('div', { style: { fontSize: 11, color: T.suave, marginTop: 'auto' } }, `${hor.length} horários · fonte: setor de transportes`)
  ])
}
''',
    'src/runtime/translations/default.ts': r'''export default {
  _widgetLabel: 'Horários da Linha'
}
''',
}

destino = os.path.join(WIDGETS_DIR, "horarios-linha")
if os.path.isdir(destino):
    shutil.copytree(destino, os.path.join(PASTA_BKP, "widget_horarios-linha_antes"), dirs_exist_ok=True)
    log("  Widget já existia — backup feito e arquivos atualizados.")
for rel, conteudo in ARQUIVOS_WIDGET.items():
    p = os.path.join(destino, *rel.split("/"))
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, "w", encoding="utf-8", newline="\n") as f:
        f.write(conteudo)
    log("  ·", p)
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — Montar o config (casca do app 2 + conteúdo de Mobilidade) =====
log("\n▶ CÉLULA 3 — Montagem do config: iniciando...")
# ---- função que copia a "casca" homologada (cabeçalho do app CCTV) para o app novo ----
import json, copy
def montar_com_casca(cctv, body, titulo="Mobilidade Urbana", mapa_novo="widget_mapa", ds_novo="dataSource_1",
                     mapa_antigo="widget_2", ds_antigo="dataSource_4", excluir=("widgets/placar-cameras/",)):
    W, L = cctv["widgets"], cctv["layouts"]
    # 1) percorre a árvore do cabeçalho
    wids, lids = [], []
    def walk(lid):
        lids.append(lid)
        for it in L[lid]["content"].values():
            wid = it.get("widgetId")
            if not wid or wid in wids: continue
            wids.append(wid)
            for lv in (W[wid].get("layouts") or {}).values():
                for l2 in set(lv.values()):
                    if l2 in L and l2 not in lids: walk(l2)
    hdr_lay = cctv["header"]["layout"]["LARGE"]
    walk(hdr_lay)
    excl = [w for w in wids if W[w]["uri"] in excluir]
    wids = [w for w in wids if w not in excl]
    sub_w = {w: copy.deepcopy(W[w]) for w in wids}
    sub_l = {l: copy.deepcopy(L[l]) for l in lids}
    for l in sub_l.values():   # tira widgets excluídos dos layouts (ex.: placar das câmeras)
        for k in [k for k, it in l["content"].items() if it.get("widgetId") in excl]:
            del l["content"][k]
            if k in l.get("order", []): l["order"].remove(k)
    for w in sub_w.values():
        b = ((w.get("config") or {}).get("behavior") or {})
        for e in excl: (b.get("size") or {}).pop(e, None)
        if w["uri"] == "widgets/arcgis/map-layers/":
            w["config"] = {"useMapWidget": True}
        if w["uri"] == "widgets/common/search/":
            # mantém só a busca de endereço (geocodificador); tira fontes ligadas às camadas do mapa do CCTV
            removidos = []
            for k, v in (w["config"].get("dataSourceConfigWithMapCentric") or {}).items():
                fica = [d for d in v.get("dataSourceConfig", []) if not d.get("useDataSource")]
                removidos += [d.get("outputDataSourceId") for d in v.get("dataSourceConfig", []) if d.get("useDataSource")]
                v["dataSourceConfig"] = fica
            for k in ("dataSourceConfig",):
                if isinstance(w["config"].get(k), list):
                    removidos += [d.get("outputDataSourceId") for d in w["config"][k] if d.get("useDataSource")]
                    w["config"][k] = [d for d in w["config"][k] if not d.get("useDataSource")]
            w["useDataSources"] = [u for u in (w.get("useDataSources") or []) if not str(u.get("dataSourceId", "")).startswith(ds_antigo + "-")]
            if not w["useDataSources"]: w.pop("useDataSources", None)
            w["outputDataSources"] = [o for o in (w.get("outputDataSources") or []) if o not in removidos]
    # 2) saídas de dados da busca + mensagens da busca
    saidas = set()
    for w in sub_w.values():
        saidas.update(w.get("outputDataSources") or [])
    out_ds = {k: copy.deepcopy(v) for k, v in cctv.get("dataSources", {}).items() if k in saidas}
    msgs = {k: copy.deepcopy(v) for k, v in cctv.get("messageConfigs", {}).items() if v.get("widgetId") in wids}
    # 3) remapeia mapa/fonte do CCTV para os do app novo
    pacote = json.dumps({"w": sub_w, "l": sub_l, "d": out_ds, "m": msgs}, ensure_ascii=False)
    pacote = pacote.replace(f"{mapa_antigo}-{ds_antigo}", f"{mapa_novo}-{ds_novo}").replace(f'"{mapa_antigo}"', f'"{mapa_novo}"')
    pacote = pacote.replace(f'"{ds_antigo}"', f'"{ds_novo}"')
    p = json.loads(pacote)
    # 4) título
    for w in p["w"].values():
        if w["uri"] == "widgets/common/text/" and "Sistema de Monitoramento" in w["config"].get("text", ""):
            w["config"]["text"] = w["config"]["text"].replace("Sistema de Monitoramento", titulo)
    # 5) monta o config final
    cfg = {k: copy.deepcopy(cctv[k]) for k in ("exbVersion", "mainSizeMode", "theme", "customTheme", "forBuilderAttributes",
                                               "template", "attributes", "sharedThemeVariables", "utilities", "mainLocale")
           if k in cctv}
    cfg["header"] = copy.deepcopy(cctv["header"])
    cfg.update({"pages": body["pages"], "pageStructure": body["pageStructure"],
                "layouts": {**p["l"], **body["layouts"]}, "widgets": {**p["w"], **body["widgets"]},
                "widgetsManifest": {}, "views": {}, "sections": {}, "dialogs": {}, "controllerPanels": {},
                "messageConfigs": {**p["m"], **body["messageConfigs"]},
                "dataSources": {**p["d"], **body["dataSources"]}})
    # conferências
    for lay in cfg["layouts"].values():
        for it in lay["content"].values():
            assert it["widgetId"] in cfg["widgets"], it["widgetId"]
    s = json.dumps(cfg)
    assert f'"{mapa_antigo}"' not in s and f"{ds_antigo}" not in s, "sobrou referência ao app de CCTV"
    return cfg, [W[w]["label"] for w in wids], [W[w]["label"] for w in excl]

BODY = json.loads(r'''{
  "pages": {
    "page_0": {
      "id": "page_0",
      "mode": "FIT_WINDOW",
      "type": "NORMAL",
      "layout": {
        "LARGE": "layout_p0"
      },
      "isDefault": true,
      "isVisible": true,
      "label": "Linhas e Horários",
      "header": true
    },
    "page_sobre": {
      "id": "page_sobre",
      "mode": "FIT_WINDOW",
      "type": "NORMAL",
      "layout": {
        "LARGE": "layout_p_sobre"
      },
      "isDefault": false,
      "isVisible": true,
      "label": "Leia-me",
      "header": true
    }
  },
  "pageStructure": [
    {
      "page_0": []
    },
    {
      "page_sobre": []
    }
  ],
  "layouts": {
    "layout_card": {
      "type": "FIXED",
      "content": {
        "0": {
          "type": "WIDGET",
          "widgetId": "widget_card_icone",
          "isPending": false,
          "bbox": {
            "left": "2%",
            "top": "12px",
            "width": "40px",
            "height": "40px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false,
            "lockParent": true
          }
        },
        "1": {
          "type": "WIDGET",
          "widgetId": "widget_card_texto",
          "isPending": false,
          "bbox": {
            "left": "14%",
            "top": "2px",
            "width": "84%",
            "height": "60px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false,
            "lockParent": true
          }
        }
      },
      "order": [
        "0",
        "1"
      ],
      "label": "Padrão"
    },
    "layout_mapa1": {
      "type": "FIXED",
      "content": {},
      "order": []
    },
    "layout_mapa_mini": {
      "type": "FIXED",
      "content": {},
      "order": []
    },
    "layout_p0": {
      "type": "FIXED",
      "content": {
        "0": {
          "type": "WIDGET",
          "widgetId": "widget_tit_lista",
          "isPending": false,
          "bbox": {
            "left": "1%",
            "top": "1%",
            "width": "24%",
            "height": "8%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "1": {
          "type": "WIDGET",
          "widgetId": "widget_lista",
          "isPending": false,
          "bbox": {
            "left": "1%",
            "top": "10%",
            "width": "24%",
            "height": "89%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "2": {
          "type": "WIDGET",
          "widgetId": "widget_mapa",
          "isPending": false,
          "bbox": {
            "left": "25.8%",
            "top": "1%",
            "width": "47.4%",
            "height": "98%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "3": {
          "type": "WIDGET",
          "widgetId": "widget_btn_leia",
          "isPending": false,
          "bbox": {
            "left": "45%",
            "top": "2%",
            "width": "9%",
            "height": "38px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "4": {
          "type": "WIDGET",
          "widgetId": "widget_horarios",
          "isPending": false,
          "bbox": {
            "left": "74%",
            "top": "1%",
            "width": "25%",
            "height": "70%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "5": {
          "type": "WIDGET",
          "widgetId": "widget_tit_calor",
          "isPending": false,
          "bbox": {
            "left": "74%",
            "top": "72%",
            "width": "25%",
            "height": "4.5%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "6": {
          "type": "WIDGET",
          "widgetId": "widget_mapa_mini",
          "isPending": false,
          "bbox": {
            "left": "74%",
            "top": "77%",
            "width": "25%",
            "height": "22%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        }
      },
      "order": [
        "0",
        "1",
        "2",
        "3",
        "4",
        "5",
        "6"
      ],
      "label": "Linhas e Horários"
    },
    "layout_p_sobre": {
      "type": "FIXED",
      "content": {
        "0": {
          "type": "WIDGET",
          "widgetId": "widget_sobre_txt",
          "isPending": false,
          "bbox": {
            "left": "25%",
            "top": "5%",
            "width": "50%",
            "height": "78%"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        },
        "1": {
          "type": "WIDGET",
          "widgetId": "widget_voltar3",
          "isPending": false,
          "bbox": {
            "left": "43.5%",
            "top": "86%",
            "width": "13%",
            "height": "40px"
          },
          "setting": {
            "autoProps": {
              "left": true,
              "right": false,
              "top": true,
              "bottom": false
            },
            "vCenter": false,
            "hCenter": false
          }
        }
      },
      "order": [
        "0",
        "1"
      ],
      "label": "Leia-me"
    }
  },
  "widgets": {
    "widget_tit_lista": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Título – lista",
      "id": "widget_tit_lista",
      "config": {
        "text": "<p style=\"text-align:center;margin:0 0 4px 0\"><span style=\"font-size:20px;font-weight:700;color:#4463a9\">Linhas de Ônibus</span></p><p style=\"text-align:center;margin:0\"><span style=\"font-size:12px;color:#3d4657\">Clique numa linha para ver o trajeto e os horários</span></p>",
        "style": {
          "verticalAlign": "center",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-1"
        }
      }
    },
    "widget_card_icone": {
      "uri": "widgets/common/image/",
      "version": "1.19.0",
      "label": "Cartão – ícone ônibus",
      "id": "widget_card_icone",
      "config": {
        "functionConfig": {
          "altText": "Ônibus",
          "toolTip": "",
          "linkParam": {},
          "scale": "Fit",
          "imageParam": {
            "url": "",
            "originalUrl": "",
            "fileName": "onibus.svg",
            "originalName": "onibus.svg",
            "imgSourceType": "BY_URL"
          },
          "imgSourceType": "",
          "imageFillMode": "FIT"
        },
        "styleConfig": {}
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1-linhas_onibus",
          "mainDataSourceId": "dataSource_1-linhas_onibus",
          "rootDataSourceId": "dataSource_1",
          "fields": []
        }
      ],
      "useDataSourcesEnabled": true
    },
    "widget_card_texto": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Cartão – nome completo da linha",
      "id": "widget_card_texto",
      "config": {
        "text": "<p style=\"margin:0;line-height:1.3\"><span style=\"font-size:14px;font-weight:700;color:#1b2a4a\"><exp data-uniqueid=\"35f34346_94d6_4218_b1fd_99b2b464a85e\" data-dsid=\"dataSource_1-linhas_onibus\" data-expression=\"%7B%22name%22%3A%22%7BCOD_LINHA%7D%22%2C%22parts%22%3A%5B%7B%22type%22%3A%22FIELD%22%2C%22exp%22%3A%22%7BCOD_LINHA%7D%22%2C%22dataSourceId%22%3A%22dataSource_1-linhas_onibus%22%2C%22jimuFieldName%22%3A%22cod_linha%22%2C%22isFromRepeatedDataSourceContext%22%3Atrue%7D%5D%7D\">﻿<span contenteditable=\"false\">﻿﻿{COD_LINHA}﻿﻿</span>﻿</exp> - <exp data-uniqueid=\"39ecf84a_9ad1_425e_a4d2_6f316deb2b9c\" data-dsid=\"dataSource_1-linhas_onibus\" data-expression=\"%7B%22name%22%3A%22%7BNOME%7D%22%2C%22parts%22%3A%5B%7B%22type%22%3A%22FIELD%22%2C%22exp%22%3A%22%7BNOME%7D%22%2C%22dataSourceId%22%3A%22dataSource_1-linhas_onibus%22%2C%22jimuFieldName%22%3A%22nome%22%2C%22isFromRepeatedDataSourceContext%22%3Atrue%7D%5D%7D\">﻿<span contenteditable=\"false\">﻿﻿{NOME}﻿﻿</span>﻿</exp></span><span style=\"font-size:13px;color:#6b7690\"> - Sentido para <exp data-uniqueid=\"6eacf8a3_a942_4752_a931_dcf1870d179c\" data-dsid=\"dataSource_1-linhas_onibus\" data-expression=\"%7B%22name%22%3A%22%7BSENTIDO%7D%22%2C%22parts%22%3A%5B%7B%22type%22%3A%22FIELD%22%2C%22exp%22%3A%22%7BSENTIDO%7D%22%2C%22dataSourceId%22%3A%22dataSource_1-linhas_onibus%22%2C%22jimuFieldName%22%3A%22sentido%22%2C%22isFromRepeatedDataSourceContext%22%3Atrue%7D%5D%7D\">﻿<span contenteditable=\"false\">﻿﻿{SENTIDO}﻿﻿</span>﻿</exp></span></p>",
        "style": {
          "verticalAlign": "center",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-1"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1-linhas_onibus",
          "mainDataSourceId": "dataSource_1-linhas_onibus",
          "rootDataSourceId": "dataSource_1",
          "fields": [
            "cod_linha",
            "nome",
            "sentido"
          ]
        }
      ],
      "useDataSourcesEnabled": true
    },
    "widget_lista": {
      "uri": "widgets/common/list/",
      "version": "1.19.0",
      "id": "widget_lista",
      "label": "Lista de linhas",
      "config": {
        "maxItems": 200,
        "space": 4,
        "verticalSpace": 4,
        "horizontalSpace": 10,
        "itemsPerPage": 20,
        "isItemStyleConfirm": true,
        "isInitialed": true,
        "pageStyle": "SCROLL",
        "itemStyle": "STYLE5",
        "hoverType": "HOVER0",
        "selectedStyle": "STYLE0",
        "differentOddEven": false,
        "scrollBarOpen": true,
        "direction": "VERTICAL",
        "alignType": "CENTER",
        "scrollStep": 2,
        "cardConfigs": {
          "DEFAULT": {
            "cardSize": {
              "LARGE": {
                "width": 421,
                "height": 64
              }
            },
            "enable": true,
            "backgroundStyle": {
              "border": {
                "border": {
                  "type": "solid",
                  "color": "",
                  "width": "0px"
                }
              },
              "borderRadius": {
                "number": [
                  8,
                  8,
                  8,
                  8
                ],
                "unit": "px"
              }
            },
            "listLayout": "Custom"
          },
          "HOVER": {
            "cardSize": {
              "LARGE": {
                "width": 421,
                "height": 64
              }
            },
            "enable": true,
            "backgroundStyle": {
              "border": {
                "border": {
                  "type": "solid",
                  "color": "",
                  "width": "0px"
                }
              },
              "borderRadius": {
                "number": [
                  8,
                  8,
                  8,
                  8
                ],
                "unit": "px"
              },
              "background": {
                "color": "#eaf0fb",
                "fillType": "fill",
                "image": {
                  "url": ""
                }
              }
            },
            "listLayout": "Auto"
          },
          "SELECTED": {
            "cardSize": {
              "LARGE": {
                "width": 421,
                "height": 64
              }
            },
            "backgroundStyle": {
              "border": {
                "borderLeft": {
                  "type": "solid",
                  "color": "#012a84",
                  "width": "4px"
                },
                "borderBottom": {
                  "type": "solid",
                  "color": "",
                  "width": "2px"
                }
              },
              "borderRadius": {
                "number": [
                  8,
                  8,
                  8,
                  8
                ],
                "unit": "px"
              },
              "background": {
                "color": "rgba(1,42,132,0.10)",
                "fillType": "fill",
                "image": {
                  "url": ""
                }
              }
            },
            "selectionMode": "SINGLE",
            "listLayout": "Auto"
          }
        },
        "style": {
          "id": "default"
        },
        "isCheckEmptyTemplate": false,
        "filters": null,
        "isShowAutoRefresh": false,
        "layoutType": "ROW",
        "keepAspectRatio": false,
        "searchOpen": true,
        "sortOpen": false,
        "showRefresh": true,
        "showSelectedOnlyOpen": false,
        "showClearSelected": true,
        "showRecordCount": false,
        "filterOpen": false,
        "sqlExprObj": null,
        "sortFields": null,
        "searchFields": [
          "id_rota",
          "nome",
          "sentido"
        ],
        "searchHint": "Nº ou nome da linha",
        "hidePageTotal": false,
        "navigatorOpen": false,
        "lockItemRatio": false
      },
      "dataActions": {
        "setFilter": {
          "enabled": true
        },
        "relatedData": {
          "enabled": true
        }
      },
      "useDataSourcesEnabled": true,
      "layouts": {
        "DEFAULT": {
          "LARGE": "layout_card"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1-linhas_onibus",
          "mainDataSourceId": "dataSource_1-linhas_onibus",
          "rootDataSourceId": "dataSource_1",
          "fields": [
            "cod_linha",
            "nome",
            "sentido",
            "id_rota"
          ]
        }
      ],
      "outputDataSources": [],
      "style": {
        "background": {
          "color": "#ffffff",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_mapa": {
      "uri": "widgets/arcgis/arcgis-map/",
      "version": "1.19.0",
      "id": "widget_mapa",
      "label": "Mapa",
      "config": {
        "toolConfig": {
          "canZoom": true,
          "canHome": true,
          "canSearch": false,
          "canNavigation": true,
          "canLocate": true,
          "canCompass": true,
          "canFullScreen": false,
          "canScaleBar": true
        },
        "isUseCustomMapState": false,
        "initialMapDataSourceID": "dataSource_1",
        "initialMapState": null,
        "scaleRange": null
      },
      "layouts": {
        "MapFixedLayout": {
          "LARGE": "layout_mapa1"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_1",
          "mainDataSourceId": "dataSource_1"
        }
      ]
    },
    "widget_btn_leia": {
      "uri": "widgets/common/button/",
      "version": "1.19.0",
      "id": "widget_btn_leia",
      "label": "ⓘ Leia-me",
      "config": {
        "functionConfig": {
          "text": "ⓘ Leia-me",
          "toolTip": "Fonte dos dados e como usar",
          "linkParam": {
            "linkType": "PAGE",
            "value": "page_sobre",
            "openType": "_self"
          }
        },
        "styleConfig": {
          "useCustom": true,
          "customStyle": {
            "regular": {
              "background": {
                "color": "var(--primary)",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#076fe5",
                "width": "0px"
              },
              "text": {
                "color": "var(--light)",
                "size": "14px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 14
              }
            },
            "hover": {
              "background": {
                "color": "var(--primary-600)",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#076fe5",
                "width": "0px"
              },
              "text": {
                "color": "var(--light)",
                "size": "14px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 14
              }
            }
          }
        }
      }
    },
    "widget_horarios": {
      "uri": "widgets/horarios-linha/",
      "version": "1.19.0",
      "id": "widget_horarios",
      "label": "Horários da Linha",
      "config": {
        "dataSourceId": "dataSource_1-linhas_onibus",
        "linhasUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Linhas_Onibus/FeatureServer/0",
        "horariosUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Horarios_Onibus/FeatureServer/0",
        "tema": "escuro"
      }
    },
    "widget_tit_calor": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Título – concentração",
      "id": "widget_tit_calor",
      "config": {
        "text": "<p style=\"margin:0\"><span style=\"font-size:14px;font-weight:700;color:#4463a9\">Concentração de pontos de ônibus</span><span style=\"font-size:11px;color:#6b7690\"> · amarelo = mais pontos próximos · use ⤢ para tela cheia</span></p>",
        "style": {
          "verticalAlign": "center",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-1"
        }
      }
    },
    "widget_mapa_mini": {
      "uri": "widgets/arcgis/arcgis-map/",
      "version": "1.19.0",
      "id": "widget_mapa_mini",
      "label": "Mapa – concentração de pontos",
      "config": {
        "toolConfig": {
          "canZoom": true,
          "canHome": true,
          "canSearch": false,
          "canNavigation": true,
          "canLocate": false,
          "canCompass": false,
          "canFullScreen": true,
          "canScaleBar": false
        },
        "isUseCustomMapState": false,
        "initialMapDataSourceID": "dataSource_2",
        "initialMapState": null,
        "scaleRange": null
      },
      "layouts": {
        "MapFixedLayout": {
          "LARGE": "layout_mapa_mini"
        }
      },
      "useDataSources": [
        {
          "dataSourceId": "dataSource_2",
          "mainDataSourceId": "dataSource_2"
        }
      ],
      "style": {
        "background": {
          "color": "#ffffff",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_sobre_txt": {
      "uri": "widgets/common/text/",
      "version": "1.19.0",
      "label": "Leia-me – texto",
      "id": "widget_sobre_txt",
      "config": {
        "text": "<p style=\"margin:0 0 10px 0\"><span style=\"font-size:22px;font-weight:700;color:#4463a9\">Leia-me</span></p><p style=\"margin:0 0 8px 0\"><span style=\"font-size:14px;color:#3d4657\">O <b>Mobilidade Urbana</b> reúne as linhas, os pontos e os horários do transporte coletivo de Apucarana num só lugar, para consulta rápida pelo cidadão.</span></p><p style=\"margin:0 0 4px 0\"><span style=\"font-size:14px;font-weight:700;color:#1b2a4a\">Como usar</span></p><p style=\"margin:0 0 8px 0\"><span style=\"font-size:13px;color:#3d4657\">Busque a linha pelo número ou nome na lista à esquerda. Ao clicar, o mapa aproxima o trajeto e a Tabela de Horários, à direita, mostra a próxima saída e todos os horários do dia. Abaixo dela, o mapa de concentração de pontos pode ser ampliado em tela cheia.</span></p><p style=\"margin:0 0 4px 0\"><span style=\"font-size:14px;font-weight:700;color:#1b2a4a\">Fonte dos dados</span></p><p style=\"margin:0 0 8px 0\"><span style=\"font-size:13px;color:#3d4657\">Itinerários, pontos e quadro de horários: setor de transportes da Prefeitura de Apucarana. Bairros e eixos de vias: base cartográfica PMAPU.</span></p><p style=\"margin:0 0 4px 0\"><span style=\"font-size:14px;font-weight:700;color:#1b2a4a\">Antes de usar</span></p><p style=\"margin:0 0 8px 0\"><span style=\"font-size:13px;color:#3d4657\">Os horários são de <b>dias úteis</b>. Quando um sentido não tem horários próprios, são mostrados os horários de saída da mesma linha nos outros sentidos.</span></p><p style=\"margin:0\"><span style=\"font-size:12px;color:#6b7690\">Elaboração: GeoAPUC · Prefeitura de Apucarana</span></p>",
        "style": {
          "verticalAlign": "start",
          "wrap": true,
          "overflow": "auto",
          "padding": "p-4"
        }
      },
      "style": {
        "background": {
          "color": "#ffffff",
          "fillType": "fill",
          "image": {
            "url": ""
          }
        },
        "borderRadius": {
          "number": [
            12,
            12,
            12,
            12
          ],
          "unit": "px"
        },
        "border": {
          "type": "solid",
          "color": "#dfe5ef",
          "width": "1px"
        }
      }
    },
    "widget_voltar3": {
      "uri": "widgets/common/button/",
      "version": "1.19.0",
      "id": "widget_voltar3",
      "label": "← Voltar às linhas",
      "config": {
        "functionConfig": {
          "text": "← Voltar às linhas",
          "toolTip": "Voltar à página principal",
          "linkParam": {
            "linkType": "PAGE",
            "value": "page_0",
            "openType": "_self"
          }
        },
        "styleConfig": {
          "useCustom": true,
          "customStyle": {
            "regular": {
              "background": {
                "color": "var(--primary)",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#076fe5",
                "width": "0px"
              },
              "text": {
                "color": "var(--light)",
                "size": "14px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 14
              }
            },
            "hover": {
              "background": {
                "color": "var(--primary-600)",
                "fillType": "fill"
              },
              "border": {
                "type": "solid",
                "color": "#076fe5",
                "width": "0px"
              },
              "text": {
                "color": "var(--light)",
                "size": "14px"
              },
              "borderRadius": {
                "number": [
                  100,
                  100,
                  100,
                  100
                ],
                "unit": "px"
              },
              "iconProps": {
                "color": "#fff",
                "size": 14
              }
            }
          }
        }
      }
    }
  },
  "messageConfigs": {
    "messageConfig_lista_zoom": {
      "id": "messageConfig_lista_zoom",
      "widgetId": "widget_lista",
      "messageType": "DATA_RECORDS_SELECTION_CHANGE",
      "actions": [
        {
          "actionId": "widget_mapa-zoomToFeature-1",
          "widgetId": "widget_mapa",
          "messageWidgetId": "widget_lista",
          "actionName": "zoomToFeature",
          "description": null,
          "version": "1.19.0",
          "config": {
            "useDataSource": {
              "dataSourceId": "dataSource_1-linhas_onibus",
              "mainDataSourceId": "dataSource_1-linhas_onibus",
              "rootDataSourceId": "dataSource_1"
            },
            "useDataSources": [
              {
                "dataSourceId": "dataSource_1-linhas_onibus",
                "mainDataSourceId": "dataSource_1-linhas_onibus",
                "rootDataSourceId": "dataSource_1"
              }
            ],
            "goToInitialMapExtentWhenSelectionCleared": true,
            "useAnyTriggerData": true
          }
        },
        {
          "actionId": "widget_mapa-flash-2",
          "widgetId": "widget_mapa",
          "messageWidgetId": "widget_lista",
          "actionName": "flash",
          "description": null,
          "version": "1.19.0",
          "config": {
            "useAnyTriggerData": true,
            "messageUseDataSource": {
              "dataSourceId": "dataSource_1-linhas_onibus",
              "mainDataSourceId": "dataSource_1-linhas_onibus",
              "rootDataSourceId": "dataSource_1"
            },
            "actionUseDataSource": null,
            "sqlExprObj": null,
            "enabledDataRelationShip": true,
            "connectionType": "SET_CUSTOM_FIELDS"
          }
        }
      ]
    }
  },
  "dataSources": {
    "dataSource_1": {
      "id": "dataSource_1",
      "type": "WEB_MAP",
      "sourceLabel": "Mobilidade Urbana",
      "itemId": "c80cb05e18a645609f8f84740078b859",
      "portalUrl": "https://geo.apucarana.pr.gov.br/portal",
      "isHidden": false,
      "childDataSourceJsons": {
        "linhas_onibus": {
          "id": "dataSource_1-linhas_onibus",
          "type": "FEATURE_LAYER",
          "sourceLabel": "Linhas de ônibus",
          "portalUrl": "https://geo.apucarana.pr.gov.br/portal",
          "itemId": "8c163a4936e143be96f69b58a2911ce8",
          "layerId": "0",
          "url": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Linhas_Onibus/FeatureServer/0"
        }
      }
    },
    "dataSource_2": {
      "id": "dataSource_2",
      "type": "WEB_MAP",
      "sourceLabel": "Mobilidade Urbana - Concentração de Pontos",
      "itemId": "9905aa8ce7834363ada99bb68ff52f84",
      "portalUrl": "https://geo.apucarana.pr.gov.br/portal",
      "isHidden": false
    }
  }
}''')

with open(CFG_MODELO, encoding="utf-8") as f:
    cctv = json.load(f)
cfg, copiados, excluidos = montar_com_casca(cctv, BODY)
log("  Casca copiada do app 2:", ", ".join(copiados))
log("  Deixados de fora:", ", ".join(excluidos) or "-")
log("  Páginas:", " | ".join(p["label"] for p in cfg["pages"].values()))
log("  Widgets:", len(cfg["widgets"]))
log("✔ CÉLULA 3 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 4 — Backup e escrita do config do app 3 =====
log("\n▶ CÉLULA 4 — Escrita no app 3: iniciando...")
app3 = os.path.join(EXB_APPS, APP_NOVO)
for nome in ("config.json", os.path.join("resources", "config", "config.json"), "info.json"):
    o = os.path.join(app3, nome)
    if os.path.isfile(o):
        d = os.path.join(PASTA_BKP, "app3_" + nome.replace(os.sep, "_"))
        shutil.copy2(o, d); log("  Backup:", d)
img_o = os.path.join(EXB_APPS, APP_MODELO, "resources", "images")
if os.path.isdir(img_o):
    shutil.copytree(img_o, os.path.join(app3, "resources", "images"), dirs_exist_ok=True)
cfg["timestamp"] = int(datetime.datetime.now().timestamp() * 1000)
with open(CFG_NOVO, "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)
log("  config.json escrito:", CFG_NOVO)
log("✔ CÉLULA 4 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 5 — Próximos passos e links =====
log("\n▶ CÉLULA 5 — Próximos passos: iniciando...")
log("  1) REINICIE o client do ExB (widget novo): feche a janela do client e rode o Iniciar_ExB.bat de novo;")
log("     espere terminar a compilação (procure 'horarios-linha' sem erros na janela do client).")
log("  2) Construtor: Ctrl+F5 → mudança pequena → Salvar → Publicar")
log("  3) Teste: clique numa linha → a Tabela de Horários mostra a próxima saída e todos os horários")
log("\n  ===== LINKS =====")
log("   Construtor (app 3): https://localhost:3001/builder/?id=3")
log("   Prévia local:       https://localhost:3001/experience/3/?locale=pt-br")
log(f"\n  Backups em: {PASTA_BKP}")
log(f"  Saída completa: {ARQ_TXT}")
log("✔ CÉLULA 5 concluída com sucesso.")
